# Auditoria de Calçadas — Treino e Exportação (YOLOv8n)

Pipeline: dataset (Roboflow, formato YOLOv8) → treino → métricas → exportação ONNX 192×192 (yoloinferApp).

Funciona local (RTX 4060) ou no Colab.

In [ ]:
import os
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
else:
    for p in [root, *root.parents]:
        data_yaml = p / 'dataset' / 'data.yaml'
        if data_yaml.exists():
            root = p
            break

os.chdir(root)
print(f'ROOT={root}')

!pip install -q ultralytics
import torch
print('CUDA:', torch.cuda.is_available())


## 1. Dataset
Baixe o export YOLOv8 do Roboflow e ajuste `DATA` para o `data.yaml`.

In [ ]:
DATA = str(root / 'dataset' / 'data.yaml')
print(f'DATA={DATA}')
print(open(DATA, encoding='utf-8').read())


## 2. Treino

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=DATA, epochs=100, imgsz=192, batch=32,
    patience=30, project='runs/detect', name='calcadas_v1',
    exist_ok=True, device=0, workers=0, plots=True,
)


## 3. Curvas de treino

In [ ]:
from IPython.display import Image, display
base = 'runs/detect/calcadas_v1'
for f in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    display(Image(f'{base}/{f}'))

## 4. Métricas no conjunto de teste

In [ ]:
best = YOLO('runs/detect/calcadas_v1/weights/best.pt')
m = best.val(data=DATA, split='test', imgsz=192, device=0, plots=True)
print(f'mAP@50={m.box.map50:.3f}  mAP@50-95={m.box.map:.3f}  P={m.box.mp:.3f}  R={m.box.mr:.3f}')
for i, n in best.names.items():
    print(f'{n:<18}{m.box.ap50[i]:.3f}')

## 5. Exportação para ONNX (yoloinferApp)

In [ ]:
path = best.export(format='onnx', imgsz=192, simplify=True, device=0)
print(path)

## 6. Teste rápido da inferência

In [ ]:
res = best.predict('dataset/test/images', conf=0.25, save=True, imgsz=192)
print('Salvo em', res[0].save_dir)